Transformations are the building blocks of every Spark pipeline. They are lazy — calling `.filter()` or `.withColumn()` records the operation in a logical plan but executes nothing until an action is triggered. This notebook covers the core narrow transformations: operations that process each partition independently with no data movement across the network.

## Transformation Toolkit

![Core Transformations](https://raw.githubusercontent.com/schemabotview/apache-spark/main/img/spark-core-transforms.svg)

All transformations on this page are **narrow** — each output partition depends on exactly one input partition. No shuffle, no stage boundary.

## Setup

In [ ]:
import os
from pyspark.sql import SparkSession
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, BooleanType,
    DecimalType, DateType, TimestampType, DoubleType,
)

spark = (
    SparkSession.builder
    .appName("CoreTransformations")
    .master("local[*]")
    .config("spark.sql.extensions", "io.delta.sql.DeltaSparkSessionExtension")
    .config("spark.sql.catalog.spark_catalog", "org.apache.spark.sql.delta.catalog.DeltaCatalog")
    .config("spark.sql.shuffle.partitions", "4")
    .getOrCreate()
)

DATA = os.path.join(os.path.dirname(os.path.abspath("08-core-transformations.ipynb")), "data")

# Load tables used in this notebook
card_txns = spark.read.schema(StructType([
    StructField("txn_id",      StringType(),       False),
    StructField("card_id",     StringType(),       False),
    StructField("customer_id", StringType(),       False),
    StructField("amount",      DecimalType(18, 2), False),
    StructField("merchant",    StringType(),       True),
    StructField("category",    StringType(),       True),
    StructField("txn_ts",      TimestampType(),    False),
    StructField("status",      StringType(),       False),
    StructField("is_fraud",    BooleanType(),      False),
])).parquet(f"{DATA}/card_transactions")

customers = spark.read.option("header", "true").schema(StructType([
    StructField("customer_id",   StringType(),    False),
    StructField("full_name",     StringType(),    False),
    StructField("email",         StringType(),    True),
    StructField("phone",         StringType(),    True),
    StructField("city",          StringType(),    True),
    StructField("state",         StringType(),    True),
    StructField("date_of_birth", DateType(),      True),
    StructField("kyc_verified",  BooleanType(),   False),
    StructField("created_at",    TimestampType(), False),
])).csv(f"{DATA}/customers")

loan_accounts = spark.read.option("multiLine", "true").schema(StructType([
    StructField("loan_id",       StringType(),       False),
    StructField("customer_id",   StringType(),       False),
    StructField("loan_type",     StringType(),       False),
    StructField("principal",     DecimalType(18, 2), False),
    StructField("interest_rate", DoubleType(),       False),
    StructField("tenure_months", IntegerType(),      False),
    StructField("disbursed_on",  DateType(),         False),
    StructField("status",        StringType(),       False),
])).json(f"{DATA}/loan_accounts")

print("Tables loaded.")
print(f"  card_transactions : {card_txns.count()} rows")
print(f"  customers         : {customers.count()} rows")
print(f"  loan_accounts     : {loan_accounts.count()} rows")

## `select` and `selectExpr`

`select` projects columns — it returns a new DataFrame with only the specified columns. `selectExpr` accepts SQL expression strings, which is convenient for inline calculations and aliases without importing functions.

In [ ]:
from pyspark.sql.functions import col

# select — pick specific columns
card_txns.select("txn_id", "customer_id", "amount", "status").show(5)

# selectExpr — SQL expressions inline
card_txns.selectExpr(
    "txn_id",
    "customer_id",
    "amount",
    "ROUND(amount * 0.02, 2)  AS processing_fee",
    "is_fraud",
).show(5)

## `filter` and `where`

`filter` and `where` are identical. Both accept a Column expression or a SQL string. When combining conditions use `&` (AND), `|` (OR), and `~` (NOT) — always wrap each condition in parentheses.

In [ ]:
from pyspark.sql.functions import col

# Single condition — flag suspicious transactions
fraud = card_txns.filter(col("is_fraud") == True)
print(f"Fraud transactions : {fraud.count()}")

# Multiple conditions — high-value fraud transactions
high_fraud = card_txns.filter(
    (col("is_fraud") == True) & (col("amount") > 5000)
)
print(f"High-value fraud   : {high_fraud.count()}")

# SQL string form — useful for quick ad-hoc checks
card_txns.where("status = 'DECLINED' AND amount > 1000").show(5)

# NOT isin — exclude test/internal merchants
card_txns.filter(
    ~col("status").isin("REVERSED", "DECLINED")
).select("txn_id", "merchant", "amount", "status").show(5)

## `withColumn`, `withColumnRenamed`, `drop`

- `withColumn(name, expr)` — adds a new column or **replaces** an existing one if `name` already exists
- `withColumnRenamed(old, new)` — renames a column without changing its values
- `drop(*cols)` — removes one or more columns

In [ ]:
from pyspark.sql.functions import col, lit, upper, when

# Add a risk_level column based on transaction amount
enriched = (
    card_txns
    .withColumn(
        "risk_level",
        when(col("amount") > 20000, "CRITICAL")
        .when(col("amount") > 5000,  "HIGH")
        .when(col("amount") > 1000,  "MEDIUM")
        .otherwise("LOW")
    )
    .withColumn("is_fraud_flag", col("is_fraud").cast("string"))  # BooleanType → StringType
    .withColumn("source", lit("card_processor"))                  # constant column
)
enriched.select("txn_id", "amount", "risk_level", "is_fraud_flag", "source").show(8)

In [ ]:
# withColumnRenamed — expose txn_ts as transaction_timestamp for downstream consumers
renamed = (
    card_txns
    .withColumnRenamed("txn_id", "transaction_id")
    .withColumnRenamed("txn_ts", "transaction_timestamp")
)
renamed.columns

In [ ]:
# drop — remove internal columns before writing to the reporting layer
clean = card_txns.drop("card_id", "is_fraud")
print("After drop:", clean.columns)

## Type Casting

`cast` converts a column to a different type. Pass a type object or a DDL string — the DDL form is usually more readable. Invalid casts produce `null` rather than raising an error.

In [ ]:
from pyspark.sql.types import DoubleType, StringType, LongType

# Cast DecimalType amount → DoubleType for ML feature vectors
card_txns.withColumn("amount_double", col("amount").cast(DoubleType())).printSchema()

# DDL string form — more concise
card_txns.select(
    "txn_id",
    col("amount").cast("double").alias("amount_d"),
    col("txn_ts").cast("date").alias("txn_date"),       # Timestamp → Date (drops time part)
    col("is_fraud").cast("integer").alias("fraud_flag"), # Boolean → 0/1
).show(5)

In [ ]:
# Cast interest_rate (DoubleType) → DecimalType for financial reporting
from pyspark.sql.types import DecimalType

loan_accounts.select(
    "loan_id",
    "interest_rate",
    col("interest_rate").cast(DecimalType(5, 2)).alias("rate_decimal"),
).show(5)

## String Functions

`pyspark.sql.functions` provides vectorised string operations that run on the JVM — always prefer these over Python UDFs for string work.

| Function | Purpose |
|---|---|
| `upper` / `lower` | Change case |
| `trim` / `ltrim` / `rtrim` | Strip whitespace |
| `substring(col, pos, len)` | Extract characters |
| `concat` / `concat_ws` | Join strings |
| `split(col, pattern)` | Split into array |
| `regexp_replace` | Replace with regex |
| `length` | String length |

In [ ]:
from pyspark.sql.functions import (
    upper, lower, trim, substring, concat_ws, split, length, regexp_replace
)

customers.select(
    "customer_id",
    upper(col("full_name")).alias("name_upper"),
    lower(col("email")).alias("email_lower"),
    substring(col("customer_id"), 5, 4).alias("cust_num"),          # "CUST0001" → "0001"
    split(col("full_name"), " ")[0].alias("first_name"),             # "Aarav Sharma" → "Aarav"
    split(col("full_name"), " ")[1].alias("last_name"),
    length(col("full_name")).alias("name_len"),
    regexp_replace(col("phone"), r"^(\d{1})(\d{4})(\d{5})$", "$1****$3").alias("masked_phone"),
).show(6, truncate=False)

In [ ]:
# concat_ws — build a display label combining city and state
from pyspark.sql.functions import concat_ws

customers.select(
    "customer_id",
    "full_name",
    concat_ws(", ", col("city"), col("state")).alias("location"),
).show(6, truncate=False)

## Date and Timestamp Functions

Spark stores dates as `DateType` (days since epoch) and timestamps as `TimestampType` (microseconds). The functions below work on both.

| Function | Purpose |
|---|---|
| `year` / `month` / `dayofmonth` | Extract date parts |
| `date_format(col, fmt)` | Format as string |
| `datediff(end, start)` | Days between two dates |
| `add_months(col, n)` | Add calendar months |
| `current_date()` | Today's date |
| `to_date(col, fmt)` | Parse string → DateType |

In [ ]:
from pyspark.sql.functions import (
    year, month, dayofmonth, date_format,
    datediff, current_date, add_months
)

customers.select(
    "customer_id",
    "full_name",
    "date_of_birth",
    year(col("date_of_birth")).alias("birth_year"),
    datediff(current_date(), col("date_of_birth")).alias("age_days"),
    (datediff(current_date(), col("date_of_birth")) / 365).cast("int").alias("age_years"),
).show(6)

In [ ]:
# Extract transaction month from card_transactions for monthly aggregations
card_txns.select(
    "txn_id",
    "txn_ts",
    year(col("txn_ts")).alias("txn_year"),
    month(col("txn_ts")).alias("txn_month"),
    date_format(col("txn_ts"), "yyyy-MM").alias("month_key"),   # "2024-03"
    col("txn_ts").cast("date").alias("txn_date"),
).show(6)

In [ ]:
# add_months — calculate loan maturity date from disbursed_on + tenure_months
loan_accounts.select(
    "loan_id",
    "disbursed_on",
    "tenure_months",
    add_months(col("disbursed_on"), col("tenure_months")).alias("maturity_date"),
    datediff(
        add_months(col("disbursed_on"), col("tenure_months")),
        current_date()
    ).alias("days_to_maturity"),
).show(6)

## `when` / `otherwise` — Conditional Columns

`when` is the DataFrame equivalent of SQL `CASE WHEN`. Chain multiple `.when()` calls and end with `.otherwise()` for the default value. Omitting `.otherwise()` leaves unmatched rows as `null`.

In [ ]:
from pyspark.sql.functions import when, col

# Classify card transactions by amount bucket
card_txns.select(
    "txn_id",
    "amount",
    when(col("amount") > 20000, "LARGE")
    .when(col("amount") > 5000,  "MEDIUM")
    .when(col("amount") > 500,   "SMALL")
    .otherwise("MICRO")
    .alias("amount_bucket"),
    when(col("is_fraud") & (col("amount") > 10000), "CRITICAL FRAUD")
    .when(col("is_fraud"),                           "FRAUD")
    .otherwise("CLEAN")
    .alias("fraud_label"),
).show(8)

In [ ]:
# Classify loan accounts by health status
loan_accounts.select(
    "loan_id",
    "status",
    "interest_rate",
    when(col("status") == "NPA",    "Non-Performing")
    .when(col("status") == "ACTIVE", "Performing")
    .when(col("status") == "CLOSED", "Closed")
    .otherwise("Unknown")
    .alias("loan_health"),
    when(col("interest_rate") > 15, "HIGH RATE")
    .when(col("interest_rate") > 10, "MED RATE")
    .otherwise("LOW RATE")
    .alias("rate_band"),
).show(8)

## Chaining Transformations

Spark transformations are designed to chain — each returns a new DataFrame. Use parentheses to split long chains across lines for readability. The entire chain is one logical plan; Catalyst optimises it as a whole before execution.

In [ ]:
from pyspark.sql.functions import (
    col, when, upper, date_format, month, year, lit
)

# Build a reporting-ready card transaction DataFrame in one chain
report = (
    card_txns
    # 1. filter — only successful transactions
    .filter(col("status") == "SUCCESS")
    # 2. cast — amount to double for downstream ML
    .withColumn("amount_d", col("amount").cast("double"))
    # 3. derive — month key for partitioned writes
    .withColumn("month_key", date_format(col("txn_ts"), "yyyy-MM"))
    # 4. classify — risk level
    .withColumn(
        "risk_level",
        when(col("amount") > 10000, "HIGH")
        .when(col("amount") > 2000,  "MEDIUM")
        .otherwise("LOW")
    )
    # 5. project — keep only reporting columns
    .select("txn_id", "customer_id", "amount_d", "month_key", "risk_level", "merchant", "category")
)

print(f"Reporting rows: {report.count()}")
report.show(8, truncate=False)

## Summary

- All transformations on this page are **narrow** — they process each partition independently with no shuffle.
- `select` / `selectExpr` project columns; `selectExpr` accepts inline SQL and aliases.
- `filter` / `where` are identical; combine conditions with `&`, `|`, `~` — always parenthesise each condition.
- `withColumn` adds or replaces; `withColumnRenamed` renames; `drop` removes.
- `cast` converts types — invalid casts produce `null` rather than errors. Prefer DDL string form (`"double"`, `"date"`) for readability.
- String functions (`upper`, `split`, `regexp_replace`, etc.) and date functions (`year`, `datediff`, `add_months`, etc.) run on the JVM — always prefer them over Python UDFs.
- `when` / `otherwise` is the DataFrame `CASE WHEN` — chain multiple conditions and always provide `.otherwise()` to avoid unexpected nulls.
- Chain transformations freely — Catalyst optimises the entire chain as one logical plan.